In [432]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\venv\Scripts\python.exe

Python version:
3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]


In [433]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

SECTION_FILE = PROCESSED_DIR / "extracted_sections_v2.json"

print("Section file:")
print(SECTION_FILE)

print("\nFile exists:", SECTION_FILE.exists())

Section file:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\data\processed\extracted_sections_v2.json

File exists: True


In [434]:
with open(SECTION_FILE, "r", encoding="utf-8") as f:
    sections = json.load(f)

print("Sections loaded:", len(sections))

Sections loaded: 13


In [435]:
sections_df = pd.DataFrame(sections)

sections_df[
    [
        "page",
        "section_number",
        "section_title",
        "level"
    ]
]

,page,section_number,section_title,level
0,1,1,System Overview,1
1,1,2,High-Level Architecture,1
2,2,3,Software Components,1
3,2,4,Component Inventory,1
4,3,5,Ports and Interfaces,1
5,3,6,Interface Definitions,1
6,4,7,Signals and Data Flow,1
7,4,8,Functional Flow,1
8,4,9,Dependency Mapping,1
9,5,10,Architecture Relationships,1


In [436]:
for section in sections:
    print("\n" + "=" * 80)
    print(
        f"SECTION {section['section_number']}: "
        f"{section['section_title']}"
    )
    print(f"PAGE: {section['page']}")
    print("=" * 80)
    print(section["text"])


SECTION 1: System Overview
PAGE: 1
The system monitors battery voltage, battery temperature and state of charge. The BatteryMonitor software component publishes battery status information. The ThermalController consumes battery temperature and generates a cooling request. The Dashboard consumes battery status for driver display. 

SECTION 2: High-Level Architecture
PAGE: 1
The application layer contains BatteryMonitor, ThermalController and Dashboard. Communication between application components is performed through defined ports and interfaces. The Runtime Environment (RTE) provides the communication mechanism between application software components. 

SECTION 3: Software Components
PAGE: 2
3.1 BatteryMonitor BatteryMonitor acquires battery measurements and calculates BatteryStatus. It provides BatteryVoltage, BatteryTemperature and BatteryStatus information to other components. 3.2 ThermalController ThermalController receives BatteryTemperature from BatteryMonitor and determines whe

In [437]:
architecture_sections = [
    "Software Components",
    "Component Inventory",
    "Ports and Interfaces",
    "Interface Definitions",
    "Signals and Data Flow",
    "Functional Flow",
    "Dependency Mapping",
    "Architecture Relationships"
]

for section in sections:
    if section["section_title"] in architecture_sections:
        print("\n" + "=" * 100)
        print(f"SECTION {section['section_number']}: {section['section_title']}")
        print(f"PAGE: {section['page']}")
        print("=" * 100)
        print(section["text"])


SECTION 3: Software Components
PAGE: 2
3.1 BatteryMonitor BatteryMonitor acquires battery measurements and calculates BatteryStatus. It provides BatteryVoltage, BatteryTemperature and BatteryStatus information to other components. 3.2 ThermalController ThermalController receives BatteryTemperature from BatteryMonitor and determines whether cooling is required. It publishes CoolingRequest to the cooling subsystem. 3.3 Dashboard Dashboard receives BatteryStatus and presents battery information to the vehicle user. 

SECTION 4: Component Inventory
PAGE: 2
Component Role Inputs Outputs BatteryMonitor Battery measurement and status BatteryVoltage, BatteryTemperature BatteryStatus ThermalController Thermal management BatteryTemperature CoolingRequest Dashboard Driver display BatteryStatus DisplayUpdate 

SECTION 5: Ports and Interfaces
PAGE: 3
5.1 BatteryMonitor Ports P_BatteryVoltage provides battery voltage information. P_BatteryTemperature provides battery temperature information. P_Batt

In [438]:
import re

component_section = None

for section in sections:
    if section["section_title"] == "Software Components":
        component_section = section
        break

if component_section is None:
    raise ValueError("Software Components section not found.")

print("Source section:")
print(component_section["section_title"])
print("Page:", component_section["page"])

Source section:
Software Components
Page: 2


In [439]:
component_pattern = re.compile(
    r"^\s*\d+\.\d+\s+([A-Za-z][A-Za-z0-9_]*)",
    re.MULTILINE
)
component_matches = component_pattern.findall(
    component_section["text"]
)

print("Components found:", len(component_matches))
print(component_matches)

Components found: 1
['BatteryMonitor']


In [440]:
components = []

for component_name in component_matches:
    components.append({
        "name": component_name,
        "type": "SoftwareComponent",
        "document": component_section["document"],
        "page": component_section["page"],
        "section_number": component_section["section_number"],
        "section_title": component_section["section_title"]
    })

components

[{'name': 'BatteryMonitor',
  'type': 'SoftwareComponent',
  'document': 'sample_autosar_hld.pdf',
  'page': 2,
  'section_number': '3',
  'section_title': 'Software Components'}]

In [441]:
components_df = pd.DataFrame(components)

components_df

,name,type,document,page,section_number,section_title
0,BatteryMonitor,SoftwareComponent,sample_autosar_hld.pdf,2,3,Software Components


In [442]:
component_inventory_section = None

for section in sections:
    if section["section_title"] == "Component Inventory":
        component_inventory_section = section
        break

if component_inventory_section is None:
    raise ValueError("Component Inventory section not found.")

print("Source section:", component_inventory_section["section_title"])
print("Page:", component_inventory_section["page"])
print()
print(component_inventory_section["text"])

Source section: Component Inventory
Page: 2

Component Role Inputs Outputs BatteryMonitor Battery measurement and status BatteryVoltage, BatteryTemperature BatteryStatus ThermalController Thermal management BatteryTemperature CoolingRequest Dashboard Driver display BatteryStatus DisplayUpdate 


In [443]:
inventory_text = component_inventory_section["text"]

inventory_pattern = re.compile(
    r"(BatteryMonitor|ThermalController|Dashboard)\s+"
    r"(.+?)\s+"
    r"(BatteryVoltage,\s+BatteryTemperature|BatteryTemperature|BatteryStatus)\s+"
    r"(BatteryStatus|CoolingRequest|DisplayUpdate)"
)

inventory_matches = inventory_pattern.findall(inventory_text)

print("Inventory records found:", len(inventory_matches))

for match in inventory_matches:
    print(match)

Inventory records found: 3
('BatteryMonitor', 'Battery measurement and status', 'BatteryVoltage, BatteryTemperature', 'BatteryStatus')
('ThermalController', 'Thermal management', 'BatteryTemperature', 'CoolingRequest')
('Dashboard', 'Driver display', 'BatteryStatus', 'DisplayUpdate')


In [444]:
component_inventory = {}

for component, role, inputs, outputs in inventory_matches:
    component_inventory[component] = {
        "role": role,
        "inputs": [
            item.strip()
            for item in inputs.split(",")
        ],
        "outputs": [
            item.strip()
            for item in outputs.split(",")
        ]
    }

component_inventory

{'BatteryMonitor': {'role': 'Battery measurement and status',
  'inputs': ['BatteryVoltage', 'BatteryTemperature'],
  'outputs': ['BatteryStatus']},
 'ThermalController': {'role': 'Thermal management',
  'inputs': ['BatteryTemperature'],
  'outputs': ['CoolingRequest']},
 'Dashboard': {'role': 'Driver display',
  'inputs': ['BatteryStatus'],
  'outputs': ['DisplayUpdate']}}

In [445]:
# Merge component inventory into component records

for name, details in component_inventory.items():

    existing = next(
        (component for component in components
         if component["name"] == name),
        None
    )

    if existing:
        existing["role"] = details["role"]
        existing["inputs"] = details["inputs"]
        existing["outputs"] = details["outputs"]

    else:
        components.append({
            "name": name,
            "type": "SoftwareComponent",
            "document": "sample_autosar_hld.pdf",
            "page": 2,
            "section_number": "3",
            "section_title": "Software Components",
            "role": details["role"],
            "inputs": details["inputs"],
            "outputs": details["outputs"]
        })

print("Updated components:")

for component in components:
    print(component)

Updated components:
{'name': 'BatteryMonitor', 'type': 'SoftwareComponent', 'document': 'sample_autosar_hld.pdf', 'page': 2, 'section_number': '3', 'section_title': 'Software Components', 'role': 'Battery measurement and status', 'inputs': ['BatteryVoltage', 'BatteryTemperature'], 'outputs': ['BatteryStatus']}
{'name': 'ThermalController', 'type': 'SoftwareComponent', 'document': 'sample_autosar_hld.pdf', 'page': 2, 'section_number': '3', 'section_title': 'Software Components', 'role': 'Thermal management', 'inputs': ['BatteryTemperature'], 'outputs': ['CoolingRequest']}
{'name': 'Dashboard', 'type': 'SoftwareComponent', 'document': 'sample_autosar_hld.pdf', 'page': 2, 'section_number': '3', 'section_title': 'Software Components', 'role': 'Driver display', 'inputs': ['BatteryStatus'], 'outputs': ['DisplayUpdate']}


In [446]:
ports_section = None

for section in sections:
    if section["section_title"] == "Ports and Interfaces":
        ports_section = section
        break

if ports_section is None:
    raise ValueError("Ports and Interfaces section not found.")

print("Source section:", ports_section["section_title"])
print("Page:", ports_section["page"])
print()
print(ports_section["text"])

Source section: Ports and Interfaces
Page: 3

5.1 BatteryMonitor Ports P_BatteryVoltage provides battery voltage information. P_BatteryTemperature provides battery temperature information. P_BatteryStatus provides calculated battery status. 5.2 ThermalController Ports R_BatteryTemperature receives battery temperature. P_CoolingRequest provides the requested cooling state. 5.3 Dashboard Ports R_BatteryStatus receives BatteryStatus. P_DisplayUpdate provides the updated display information. 


In [447]:
import re

port_pattern = re.compile(
    r"(\d+\.\d+)\s+([A-Za-z0-9_]+)\s+Ports\s+"
    r"(.*?)(?=\d+\.\d+\s+[A-Za-z0-9_]+\s+Ports|$)"
)

port_sections = port_pattern.findall(ports_section["text"])

print("Port subsections found:", len(port_sections))

for subsection in port_sections:
    print("\nSection:", subsection[0])
    print("Component:", subsection[1])
    print("Text:", subsection[2].strip())

Port subsections found: 3

Section: 5.1
Component: BatteryMonitor
Text: P_BatteryVoltage provides battery voltage information. P_BatteryTemperature provides battery temperature information. P_BatteryStatus provides calculated battery status.

Section: 5.2
Component: ThermalController
Text: R_BatteryTemperature receives battery temperature. P_CoolingRequest provides the requested cooling state.

Section: 5.3
Component: Dashboard
Text: R_BatteryStatus receives BatteryStatus. P_DisplayUpdate provides the updated display information.


In [448]:
ports = []

for subsection_number, component_name, subsection_text in port_sections:

    matches = re.findall(
        r"\b([PR]_[A-Za-z0-9_]+)\s+(provides|receives)\s+(.+?)(?=\s+[PR]_[A-Za-z0-9_]+\s+(?:provides|receives)|$)",
        subsection_text.strip()
    )

    for port_name, direction_word, description in matches:

        direction = "Provider" if direction_word == "provides" else "Receiver"

        ports.append({
            "name": port_name,
            "component": component_name,
            "direction": direction,
            "description": description.strip(),
            "document": ports_section["document"],
            "page": ports_section["page"],
            "section_number": ports_section["section_number"],
            "section_title": ports_section["section_title"],
            "subsection_number": subsection_number
        })

print("Ports extracted:", len(ports))

for port in ports:
    print(port)

Ports extracted: 7
{'name': 'P_BatteryVoltage', 'component': 'BatteryMonitor', 'direction': 'Provider', 'description': 'battery voltage information.', 'document': 'sample_autosar_hld.pdf', 'page': 3, 'section_number': '5', 'section_title': 'Ports and Interfaces', 'subsection_number': '5.1'}
{'name': 'P_BatteryTemperature', 'component': 'BatteryMonitor', 'direction': 'Provider', 'description': 'battery temperature information.', 'document': 'sample_autosar_hld.pdf', 'page': 3, 'section_number': '5', 'section_title': 'Ports and Interfaces', 'subsection_number': '5.1'}
{'name': 'P_BatteryStatus', 'component': 'BatteryMonitor', 'direction': 'Provider', 'description': 'calculated battery status.', 'document': 'sample_autosar_hld.pdf', 'page': 3, 'section_number': '5', 'section_title': 'Ports and Interfaces', 'subsection_number': '5.1'}
{'name': 'R_BatteryTemperature', 'component': 'ThermalController', 'direction': 'Receiver', 'description': 'battery temperature.', 'document': 'sample_autosa

In [449]:
ports_df = pd.DataFrame(ports)

ports_df[
    [
        "name",
        "component",
        "direction",
        "description",
        "page",
        "subsection_number"
    ]
]

,name,component,direction,description,page,subsection_number
0,P_BatteryVoltage,BatteryMonitor,Provider,battery voltage information.,3,5.1
1,P_BatteryTemperature,BatteryMonitor,Provider,battery temperature information.,3,5.1
2,P_BatteryStatus,BatteryMonitor,Provider,calculated battery status.,3,5.1
3,R_BatteryTemperature,ThermalController,Receiver,battery temperature.,3,5.2
4,P_CoolingRequest,ThermalController,Provider,the requested cooling state.,3,5.2
5,R_BatteryStatus,Dashboard,Receiver,BatteryStatus.,3,5.3
6,P_DisplayUpdate,Dashboard,Provider,the updated display information.,3,5.3


In [450]:
interface_section = None

for section in sections:
    if section["section_title"] == "Interface Definitions":
        interface_section = section
        break

if interface_section is None:
    raise ValueError("Interface Definitions section not found.")

print("Source section:", interface_section["section_title"])
print("Page:", interface_section["page"])
print()
print(interface_section["text"])

Source section: Interface Definitions
Page: 3

Interface Data Elements Provider Consumer BatteryMeasurementInterface BatteryVoltage, BatteryTemperature BatteryMonitor ThermalController BatteryStatusInterface BatteryStatus BatteryMonitor Dashboard CoolingRequestInterface CoolingRequest ThermalController CoolingSystem 


In [451]:
interface_text = interface_section["text"]

interface_pattern = re.compile(
    r"(BatteryMeasurementInterface|BatteryStatusInterface|CoolingRequestInterface)\s+"
    r"(.+?)\s+"
    r"(BatteryMonitor|ThermalController)\s+"
    r"(ThermalController|Dashboard|CoolingSystem)"
)

interface_matches = interface_pattern.findall(interface_text)

print("Interface records found:", len(interface_matches))

for match in interface_matches:
    print(match)

Interface records found: 3
('BatteryMeasurementInterface', 'BatteryVoltage, BatteryTemperature', 'BatteryMonitor', 'ThermalController')
('BatteryStatusInterface', 'BatteryStatus', 'BatteryMonitor', 'Dashboard')
('CoolingRequestInterface', 'CoolingRequest', 'ThermalController', 'CoolingSystem')


In [452]:
interfaces = []

for interface_name, data_elements, provider, consumer in interface_matches:

    interfaces.append({
        "name": interface_name,
        "type": "Interface",
        "data_elements": [
            item.strip()
            for item in data_elements.split(",")
        ],
        "provider": provider,
        "consumer": consumer,
        "document": interface_section["document"],
        "page": interface_section["page"],
        "section_number": interface_section["section_number"],
        "section_title": interface_section["section_title"]
    })

print("Interfaces extracted:", len(interfaces))

for interface in interfaces:
    print(interface)

Interfaces extracted: 3
{'name': 'BatteryMeasurementInterface', 'type': 'Interface', 'data_elements': ['BatteryVoltage', 'BatteryTemperature'], 'provider': 'BatteryMonitor', 'consumer': 'ThermalController', 'document': 'sample_autosar_hld.pdf', 'page': 3, 'section_number': '6', 'section_title': 'Interface Definitions'}
{'name': 'BatteryStatusInterface', 'type': 'Interface', 'data_elements': ['BatteryStatus'], 'provider': 'BatteryMonitor', 'consumer': 'Dashboard', 'document': 'sample_autosar_hld.pdf', 'page': 3, 'section_number': '6', 'section_title': 'Interface Definitions'}
{'name': 'CoolingRequestInterface', 'type': 'Interface', 'data_elements': ['CoolingRequest'], 'provider': 'ThermalController', 'consumer': 'CoolingSystem', 'document': 'sample_autosar_hld.pdf', 'page': 3, 'section_number': '6', 'section_title': 'Interface Definitions'}


In [453]:
interfaces_df = pd.DataFrame(interfaces)

interfaces_df[
    [
        "name",
        "data_elements",
        "provider",
        "consumer",
        "page"
    ]
]

,name,data_elements,provider,consumer,page
0,BatteryMeasurementInterface,"[BatteryVoltage, BatteryTemperature]",BatteryMonitor,ThermalController,3
1,BatteryStatusInterface,[BatteryStatus],BatteryMonitor,Dashboard,3
2,CoolingRequestInterface,[CoolingRequest],ThermalController,CoolingSystem,3


In [454]:
signal_section = None

for section in sections:
    if section["section_title"] == "Signals and Data Flow":
        signal_section = section
        break

if signal_section is None:
    raise ValueError("Signals and Data Flow section not found.")

print("Source section:", signal_section["section_title"])
print("Page:", signal_section["page"])
print()
print(signal_section["text"])

Source section: Signals and Data Flow
Page: 4

BatteryVoltage and BatteryTemperature are measurement data produced by BatteryMonitor. BatteryStatus is derived from battery measurements and is consumed by Dashboard. BatteryTemperature is consumed by ThermalController. CoolingRequest is generated by ThermalController. 


In [455]:
data_flow_text = signal_section["text"]

data_flows = []

# Valid signals/data elements already extracted from the architecture
valid_signals = set()

for interface in interfaces:
    valid_signals.update(interface["data_elements"])

for component in components:
    valid_signals.update(component["inputs"])
    valid_signals.update(component["outputs"])

print("Valid signals:")
print(sorted(valid_signals))


# 1. Measurement data produced by a component
pattern = re.compile(
    r"([A-Za-z0-9_]+)\s+and\s+([A-Za-z0-9_]+)"
    r"\s+are measurement data produced by\s+([A-Za-z0-9_]+)"
)

for signal1, signal2, component in pattern.findall(data_flow_text):

    for signal in [signal1, signal2]:

        if signal in valid_signals:
            data_flows.append({
                "signal": signal,
                "relationship": "produced",
                "component": component
            })


# 2. Derived signal consumed by a component
pattern = re.compile(
    r"([A-Za-z0-9_]+)"
    r"\s+is derived from battery measurements and is consumed by\s+"
    r"([A-Za-z0-9_]+)"
)

for signal, component in pattern.findall(data_flow_text):

    if signal in valid_signals:
        data_flows.append({
            "signal": signal,
            "relationship": "consumed",
            "component": component
        })


# 3. Signal consumed by a component
pattern = re.compile(
    r"([A-Za-z0-9_]+)\s+is consumed by\s+([A-Za-z0-9_]+)"
)

for signal, component in pattern.findall(data_flow_text):

    if signal in valid_signals:
        data_flows.append({
            "signal": signal,
            "relationship": "consumed",
            "component": component
        })


# 4. Signal generated by a component
pattern = re.compile(
    r"([A-Za-z0-9_]+)\s+is generated by\s+([A-Za-z0-9_]+)"
)

for signal, component in pattern.findall(data_flow_text):

    if signal in valid_signals:
        data_flows.append({
            "signal": signal,
            "relationship": "generated",
            "component": component
        })


print("\nData-flow records:", len(data_flows))

for flow in data_flows:
    print(flow)

Valid signals:
['BatteryStatus', 'BatteryTemperature', 'BatteryVoltage', 'CoolingRequest', 'DisplayUpdate']

Data-flow records: 5
{'signal': 'BatteryVoltage', 'relationship': 'produced', 'component': 'BatteryMonitor'}
{'signal': 'BatteryTemperature', 'relationship': 'produced', 'component': 'BatteryMonitor'}
{'signal': 'BatteryStatus', 'relationship': 'consumed', 'component': 'Dashboard'}
{'signal': 'BatteryTemperature', 'relationship': 'consumed', 'component': 'ThermalController'}
{'signal': 'CoolingRequest', 'relationship': 'generated', 'component': 'ThermalController'}


In [456]:
for flow in data_flows:
    flow.update({
        "document": signal_section["document"],
        "page": signal_section["page"],
        "section_number": signal_section["section_number"],
        "section_title": signal_section["section_title"]
    })

data_flow_df = pd.DataFrame(data_flows)

data_flow_df

,signal,relationship,component,document,page,section_number,section_title
0,BatteryVoltage,produced,BatteryMonitor,sample_autosar_hld.pdf,4,7,Signals and Data Flow
1,BatteryTemperature,produced,BatteryMonitor,sample_autosar_hld.pdf,4,7,Signals and Data Flow
2,BatteryStatus,consumed,Dashboard,sample_autosar_hld.pdf,4,7,Signals and Data Flow
3,BatteryTemperature,consumed,ThermalController,sample_autosar_hld.pdf,4,7,Signals and Data Flow
4,CoolingRequest,generated,ThermalController,sample_autosar_hld.pdf,4,7,Signals and Data Flow


In [457]:
functional_flow_section = None

for section in sections:
    if section["section_title"] == "Functional Flow":
        functional_flow_section = section
        break

if functional_flow_section is None:
    raise ValueError("Functional Flow section not found.")

print("Source section:", functional_flow_section["section_title"])
print("Page:", functional_flow_section["page"])
print()
print(functional_flow_section["text"])

Source section: Functional Flow
Page: 4

1. BatteryMonitor reads battery voltage and temperature. 2. BatteryMonitor calculates BatteryStatus. 3. BatteryMonitor provides BatteryTemperature to ThermalController. 4. ThermalController evaluates the temperature and generates CoolingRequest. 5. BatteryMonitor provides BatteryStatus to Dashboard. 6. Dashboard updates the driver display. 


In [458]:
functional_flow_text = functional_flow_section["text"]

flow_pattern = re.compile(
    r"(\d+)\.\s+(.*?)(?=\s+\d+\.\s+|$)"
)

flow_matches = flow_pattern.findall(functional_flow_text)

print("Functional flow steps:", len(flow_matches))

for step_number, description in flow_matches:
    print(f"{step_number}. {description}")

Functional flow steps: 6
1. BatteryMonitor reads battery voltage and temperature.
2. BatteryMonitor calculates BatteryStatus.
3. BatteryMonitor provides BatteryTemperature to ThermalController.
4. ThermalController evaluates the temperature and generates CoolingRequest.
5. BatteryMonitor provides BatteryStatus to Dashboard.
6. Dashboard updates the driver display. 


In [459]:
functional_flows = []

for step_number, description in flow_matches:

    functional_flows.append({
        "step": int(step_number),
        "description": description.strip(),
        "document": functional_flow_section["document"],
        "page": functional_flow_section["page"],
        "section_number": functional_flow_section["section_number"],
        "section_title": functional_flow_section["section_title"]
    })

print("Functional-flow records:", len(functional_flows))

for flow in functional_flows:
    print(flow)

Functional-flow records: 6
{'step': 1, 'description': 'BatteryMonitor reads battery voltage and temperature.', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '8', 'section_title': 'Functional Flow'}
{'step': 2, 'description': 'BatteryMonitor calculates BatteryStatus.', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '8', 'section_title': 'Functional Flow'}
{'step': 3, 'description': 'BatteryMonitor provides BatteryTemperature to ThermalController.', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '8', 'section_title': 'Functional Flow'}
{'step': 4, 'description': 'ThermalController evaluates the temperature and generates CoolingRequest.', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '8', 'section_title': 'Functional Flow'}
{'step': 5, 'description': 'BatteryMonitor provides BatteryStatus to Dashboard.', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '8', 'section_title': 'Functional Flow

In [460]:
functional_flow_df = pd.DataFrame(functional_flows)

functional_flow_df

,step,description,document,page,section_number,section_title
0,1,BatteryMonitor reads battery voltage and tempe...,sample_autosar_hld.pdf,4,8,Functional Flow
1,2,BatteryMonitor calculates BatteryStatus.,sample_autosar_hld.pdf,4,8,Functional Flow
2,3,BatteryMonitor provides BatteryTemperature to ...,sample_autosar_hld.pdf,4,8,Functional Flow
3,4,ThermalController evaluates the temperature an...,sample_autosar_hld.pdf,4,8,Functional Flow
4,5,BatteryMonitor provides BatteryStatus to Dashb...,sample_autosar_hld.pdf,4,8,Functional Flow
5,6,Dashboard updates the driver display.,sample_autosar_hld.pdf,4,8,Functional Flow


In [461]:
dependency_section = None

for section in sections:
    if section["section_title"] == "Dependency Mapping":
        dependency_section = section
        break

if dependency_section is None:
    raise ValueError("Dependency Mapping section not found.")

print("Source section:", dependency_section["section_title"])
print("Page:", dependency_section["page"])
print()
print(dependency_section["text"])

Source section: Dependency Mapping
Page: 4

ThermalController depends on BatteryMonitor for BatteryTemperature. Dashboard depends on BatteryMonitor for BatteryStatus. CoolingSystem depends on ThermalController for CoolingRequest. 


In [462]:
dependency_text = dependency_section["text"]

dependency_pattern = re.compile(
    r"([A-Za-z0-9_]+)\s+depends on\s+([A-Za-z0-9_]+)"
    r"\s+for\s+([A-Za-z0-9_]+)"
)

dependency_matches = dependency_pattern.findall(dependency_text)

print("Dependency records found:", len(dependency_matches))

for match in dependency_matches:
    print(match)

Dependency records found: 3
('ThermalController', 'BatteryMonitor', 'BatteryTemperature')
('Dashboard', 'BatteryMonitor', 'BatteryStatus')
('CoolingSystem', 'ThermalController', 'CoolingRequest')


In [463]:
dependencies = []

for dependent, dependency, signal in dependency_matches:

    dependencies.append({
        "dependent_component": dependent,
        "dependency_component": dependency,
        "data_element": signal,
        "document": dependency_section["document"],
        "page": dependency_section["page"],
        "section_number": dependency_section["section_number"],
        "section_title": dependency_section["section_title"]
    })

print("Dependencies extracted:", len(dependencies))

for dependency in dependencies:
    print(dependency)

Dependencies extracted: 3
{'dependent_component': 'ThermalController', 'dependency_component': 'BatteryMonitor', 'data_element': 'BatteryTemperature', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '9', 'section_title': 'Dependency Mapping'}
{'dependent_component': 'Dashboard', 'dependency_component': 'BatteryMonitor', 'data_element': 'BatteryStatus', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '9', 'section_title': 'Dependency Mapping'}
{'dependent_component': 'CoolingSystem', 'dependency_component': 'ThermalController', 'data_element': 'CoolingRequest', 'document': 'sample_autosar_hld.pdf', 'page': 4, 'section_number': '9', 'section_title': 'Dependency Mapping'}


In [464]:
dependencies_df = pd.DataFrame(dependencies)

dependencies_df

,dependent_component,dependency_component,data_element,document,page,section_number,section_title
0,ThermalController,BatteryMonitor,BatteryTemperature,sample_autosar_hld.pdf,4,9,Dependency Mapping
1,Dashboard,BatteryMonitor,BatteryStatus,sample_autosar_hld.pdf,4,9,Dependency Mapping
2,CoolingSystem,ThermalController,CoolingRequest,sample_autosar_hld.pdf,4,9,Dependency Mapping


In [465]:
relationship_section = None

for section in sections:
    if section["section_title"] == "Architecture Relationships":
        relationship_section = section
        break

if relationship_section is None:
    raise ValueError("Architecture Relationships section not found.")

print("Source section:", relationship_section["section_title"])
print("Page:", relationship_section["page"])
print()
print(relationship_section["text"])

Source section: Architecture Relationships
Page: 5

The primary application relationships are: • BatteryMonitor → ThermalController through BatteryMeasurementInterface • BatteryMonitor → Dashboard through BatteryStatusInterface • ThermalController → CoolingSystem through CoolingRequestInterface 


In [466]:
relationship_text = relationship_section["text"]

relationship_pattern = re.compile(
    r"•\s*([A-Za-z0-9_]+)\s+→\s+([A-Za-z0-9_]+)"
    r"\s+through\s+([A-Za-z0-9_]+)"
)

relationship_matches = relationship_pattern.findall(relationship_text)

print("Architecture relationships found:", len(relationship_matches))

for match in relationship_matches:
    print(match)

Architecture relationships found: 3
('BatteryMonitor', 'ThermalController', 'BatteryMeasurementInterface')
('BatteryMonitor', 'Dashboard', 'BatteryStatusInterface')
('ThermalController', 'CoolingSystem', 'CoolingRequestInterface')


In [467]:
architecture_relationships = []

for source_component, target_component, interface_name in relationship_matches:

    architecture_relationships.append({
        "source_component": source_component,
        "target_component": target_component,
        "interface": interface_name,
        "document": relationship_section["document"],
        "page": relationship_section["page"],
        "section_number": relationship_section["section_number"],
        "section_title": relationship_section["section_title"]
    })

print(
    "Architecture relationships extracted:",
    len(architecture_relationships)
)

for relationship in architecture_relationships:
    print(relationship)

Architecture relationships extracted: 3
{'source_component': 'BatteryMonitor', 'target_component': 'ThermalController', 'interface': 'BatteryMeasurementInterface', 'document': 'sample_autosar_hld.pdf', 'page': 5, 'section_number': '10', 'section_title': 'Architecture Relationships'}
{'source_component': 'BatteryMonitor', 'target_component': 'Dashboard', 'interface': 'BatteryStatusInterface', 'document': 'sample_autosar_hld.pdf', 'page': 5, 'section_number': '10', 'section_title': 'Architecture Relationships'}
{'source_component': 'ThermalController', 'target_component': 'CoolingSystem', 'interface': 'CoolingRequestInterface', 'document': 'sample_autosar_hld.pdf', 'page': 5, 'section_number': '10', 'section_title': 'Architecture Relationships'}


In [468]:
relationships_df = pd.DataFrame(architecture_relationships)

relationships_df

,source_component,target_component,interface,document,page,section_number,section_title
0,BatteryMonitor,ThermalController,BatteryMeasurementInterface,sample_autosar_hld.pdf,5,10,Architecture Relationships
1,BatteryMonitor,Dashboard,BatteryStatusInterface,sample_autosar_hld.pdf,5,10,Architecture Relationships
2,ThermalController,CoolingSystem,CoolingRequestInterface,sample_autosar_hld.pdf,5,10,Architecture Relationships


In [469]:
architecture_knowledge = {
    "document": component_section["document"],
    "components": components,
    "ports": ports,
    "interfaces": interfaces,
    "data_flows": data_flows,
    "functional_flows": functional_flows,
    "dependencies": dependencies,
    "architecture_relationships": architecture_relationships
}

print("Architecture knowledge created.")
print("Keys:")

for key in architecture_knowledge:
    print("-", key)

Architecture knowledge created.
Keys:
- document
- components
- ports
- interfaces
- data_flows
- functional_flows
- dependencies
- architecture_relationships


In [470]:
print("Components:", len(architecture_knowledge["components"]))
print("Ports:", len(architecture_knowledge["ports"]))
print("Interfaces:", len(architecture_knowledge["interfaces"]))
print("Data flows:", len(architecture_knowledge["data_flows"]))
print("Functional flows:", len(architecture_knowledge["functional_flows"]))
print("Dependencies:", len(architecture_knowledge["dependencies"]))
print(
    "Architecture relationships:",
    len(architecture_knowledge["architecture_relationships"])
)

Components: 3
Ports: 7
Interfaces: 3
Data flows: 5
Functional flows: 6
Dependencies: 3
Architecture relationships: 3


In [471]:
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "extracted_entities"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ARCHITECTURE_FILE = OUTPUT_DIR / "architecture_knowledge.json"

with open(
    ARCHITECTURE_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        architecture_knowledge,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved successfully:")
print(ARCHITECTURE_FILE)

Saved successfully:
c:\Users\preet\OneDrive\Desktop\autosar_hld_assistant\outputs\extracted_entities\architecture_knowledge.json


In [472]:
with open(
    ARCHITECTURE_FILE,
    "r",
    encoding="utf-8"
) as f:

    loaded_architecture = json.load(f)

print("JSON loaded successfully.")
print("Document:", loaded_architecture["document"])
print("Components:", len(loaded_architecture["components"]))
print("Ports:", len(loaded_architecture["ports"]))
print("Interfaces:", len(loaded_architecture["interfaces"]))

JSON loaded successfully.
Document: sample_autosar_hld.pdf
Components: 3
Ports: 7
Interfaces: 3
